# MODEL 2: FASTER R-CNN (FPN BACKBONE) - VISDRONE STANDARD BENCHMARK

### Cơ Sở Lý Thuyết & Động Lực Lựa Chọn (Theoretical Motivation)
* **Kiến trúc:** Faster R-CNN là đại diện tiêu biểu của trường phái **Two-Stage Object Detector**. Kiến trúc gồm 2 giai đoạn tách biệt:
  1. *Stage 1 (Region Proposal Network - RPN):* Quét toàn bộ Feature Map từ Feature Pyramid Network (FPN) để sinh ra hàng ngàn vùng ứng viên tiềm năng (RoI Candidates) với đa dạng tỷ lệ khung hình (Aspect Ratios).
  2. *Stage 2 (Fast R-CNN Head & RoIAlign):* Trích xuất đặc trưng pixel chính xác tại từng RoI để phân loại lớp và tinh chỉnh hộp giới hạn (Bounding Box Regression).
* **Lý do lựa chọn cho dữ liệu Drone:** Trong repo chuẩn VisDrone, Faster R-CNN với backbone ResNet50-FPN luôn được dùng làm thước đo chuẩn mực (Standard Benchmark) nhờ khả năng giữ lại độ nhạy phát hiện (Recall) cao vượt trội đối với các vật thể nhỏ và bị che khuất một phần.
* **Đánh giá Trade-off dự kiến:** Mô hình dự kiến mang lại chất lượng Bounding Box ổn định và ít rung lắc hơn YOLO26n, nhưng tốc độ xử lý (FPS) sẽ giảm đáng kể do chi phí tính toán hai giai đoạn.

In [1]:
import os
import sys
import json
import time
import torch
import cv2
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Thiết lập đường dẫn Root
ROOT_DIR = Path("/workspace/SurvivalBuddy").resolve()
MODEL_DIR = ROOT_DIR / "models" / "02_visdrone_faster_rcnn"
RESULTS_DIR = MODEL_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(ROOT_DIR / "src") not in sys.path:
    sys.path.append(str(ROOT_DIR / "src"))

from metrics import HardwareProfiler, calculate_st_iou

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Sẵn sàng chạy Model 2 trên: {DEVICE} ({torch.cuda.get_device_name(0)})")

# Custom Dataset đọc trực tiếp từ yolo_dataset
class DroneDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, transforms=None):
        self.img_dir = Path(img_dir)
        self.lbl_dir = Path(lbl_dir)
        self.img_files = sorted(list(self.img_dir.glob("*.jpg")))
        self.transforms = transforms

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        lbl_path = self.lbl_dir / (img_path.stem + ".txt")
        
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        
        boxes = []
        labels = []
        if lbl_path.exists():
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cx, cy, bw, bh = map(float, parts[1:5])
                        x1 = (cx - bw / 2) * w
                        y1 = (cy - bh / 2) * h
                        x2 = (cx + bw / 2) * w
                        y2 = (cy + bh / 2) * h
                        if x2 > x1 and y2 > y1:
                            boxes.append([x1, y1, x2, y2])
                            labels.append(1) # Class 1: Target Object

        if not boxes:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}
        
        # Chuyển đổi ảnh sang tensor [0, 1]
        img_tensor = torchvision.transforms.functional.to_tensor(img)
        return img_tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

# Khởi tạo DataLoader
YOLO_DATA_DIR = ROOT_DIR / "dataset" / "yolo_dataset"
train_dataset = DroneDataset(YOLO_DATA_DIR / "images" / "train", YOLO_DATA_DIR / "labels" / "train")
val_dataset = DroneDataset(YOLO_DATA_DIR / "images" / "val", YOLO_DATA_DIR / "labels" / "val")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=4, collate_fn=collate_fn)

print(f"Đã nạp thành công: {len(train_dataset)} Train Samples | {len(val_dataset)} Val Samples")

Sẵn sàng chạy Model 2 trên: cuda:0 (NVIDIA GeForce RTX 3090)
Đã nạp thành công: 13411 Train Samples | 6695 Val Samples


In [4]:
import time
import torch
from tqdm.auto import tqdm
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# 1. Khởi tạo mô hình Faster R-CNN ResNet50 FPN V2 Pretrained
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn_v2(weights=weights)

# Thay thế Head phân loại cho 2 lớp (0: Background, 1: Target Object)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
model.to(DEVICE)

# 2. Cấu hình Optimizer & Scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.33)

# 3. Tiến trình Huấn luyện (15 Epochs kèm Thanh Tiến Trình tqdm)
NUM_EPOCHS = 15
print(f"Bắt đầu huấn luyện Faster R-CNN ({NUM_EPOCHS} Epochs trên RTX 3090)...")

best_loss = float("inf")
scaler = torch.amp.GradScaler('cuda')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    start_ep = time.time()
    
    # Khởi tạo thanh progress bar trực quan cho từng epoch
    pbar = tqdm(
        train_loader, 
        desc=f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}]", 
        leave=True,
        dynamic_ncols=True
    )
    
    for images, targets in pbar:
        images = list(img.to(DEVICE) for img in images)
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
        scaler.scale(losses).backward()
        scaler.step(optimizer)
        scaler.update()
        
        current_loss = losses.item()
        epoch_loss += current_loss
        
        # Cập nhật thông tin chi tiết lên progress bar
        gpu_mem = f"{torch.cuda.memory_reserved(0) / 1024**3:.2f}GB" if torch.cuda.is_available() else "N/A"
        pbar.set_postfix({
            "Loss": f"{current_loss:.4f}",
            "AvgLoss": f"{epoch_loss / (pbar.n + 1):.4f}",
            "GPU_mem": gpu_mem
        })
        
    lr_scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    elapsed = time.time() - start_ep
    
    # Lưu trọng số tốt nhất
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), RESULTS_DIR / "best_faster_rcnn.pth")
        print(f"   ↳ Epoch {epoch:02d}: Đã lưu checkpoint mới tốt nhất (Avg Loss: {avg_loss:.4f})")

print("\n" + "=" * 60)
print(f"HUẤN LUYỆN HOÀN TẤT VÀ LƯU WEIGHTS THÀNH CÔNG!")
print(f"Trọng số tốt nhất tại: {RESULTS_DIR / 'best_faster_rcnn.pth'}")
print("=" * 60)

Bắt đầu huấn luyện Faster R-CNN (15 Epochs trên RTX 3090)...


Epoch [01/15]: 100%|██████████| 3353/3353 [13:38<00:00,  4.09it/s, Loss=0.0554, AvgLoss=0.0596, GPU_mem=7.54GB]


   ↳ Epoch 01: Đã lưu checkpoint mới tốt nhất (Avg Loss: 0.0596)


Epoch [02/15]: 100%|██████████| 3353/3353 [13:41<00:00,  4.08it/s, Loss=0.0303, AvgLoss=0.0464, GPU_mem=7.54GB]


   ↳ Epoch 02: Đã lưu checkpoint mới tốt nhất (Avg Loss: 0.0464)


...

   ↳ Epoch 13: Đã lưu checkpoint mới tốt nhất (Avg Loss: 0.0144)


Epoch [14/15]: 100%|██████████| 3353/3353 [13:45<00:00,  4.06it/s, Loss=0.0118, AvgLoss=0.0138, GPU_mem=7.54GB]


   ↳ Epoch 14: Đã lưu checkpoint mới tốt nhất (Avg Loss: 0.0138)


Epoch [15/15]: 100%|██████████| 3353/3353 [13:45<00:00,  4.06it/s, Loss=0.0103, AvgLoss=0.0133, GPU_mem=7.54GB]


   ↳ Epoch 15: Đã lưu checkpoint mới tốt nhất (Avg Loss: 0.0133)

HUẤN LUYỆN HOÀN TẤT VÀ LƯU WEIGHTS THÀNH CÔNG!
Trọng số tốt nhất tại: /workspace/SurvivalBuddy/models/02_visdrone_faster_rcnn/results/best_faster_rcnn.pth


In [6]:
import os
import sys
import json
import random
import cv2
import numpy as np
import torch
import torchvision
from pathlib import Path
from tqdm.auto import tqdm
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

ROOT_DIR = Path("/workspace/SurvivalBuddy").resolve()
MODEL_DIR = ROOT_DIR / "models" / "02_visdrone_faster_rcnn"
RESULTS_DIR = MODEL_DIR / "results"

if str(ROOT_DIR / "src") not in sys.path:
    sys.path.append(str(ROOT_DIR / "src"))

from metrics import HardwareProfiler, calculate_st_iou

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 1. Nap Model Danh Gia
eval_model = fasterrcnn_resnet50_fpn_v2(weights=None)
in_features = eval_model.roi_heads.box_predictor.cls_score.in_features
eval_model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)
eval_model.load_state_dict(torch.load(RESULTS_DIR / "best_faster_rcnn.pth"))
eval_model.to(DEVICE)
eval_model.eval()

# 2. Danh gia Validation STIoU
ANNO_FILE = ROOT_DIR / "dataset" / "train" / "annotations" / "annotations.json"
SAMPLES_DIR = ROOT_DIR / "dataset" / "train" / "samples"

with open(ANNO_FILE, "r", encoding="utf-8") as f:
    raw_annotations = json.load(f)

random.seed(42)
video_ids = [v["video_id"] for v in raw_annotations if "video_id" in v]
random.shuffle(video_ids)
val_video_ids = set(video_ids[max(1, int(len(video_ids) * 0.8)):])

print(f"Bắt đầu đánh giá STIoU trên các video Validation: {val_video_ids}")
st_iou_results = {}

for v_entry in raw_annotations:
    v_id = v_entry.get("video_id")
    if v_id not in val_video_ids:
        continue
        
    gt_dict = {}
    for anno in v_entry.get("annotations", []):
        for item in anno.get("bboxes", []):
            f_num = item.get("frame")
            x1, y1, x2, y2 = item.get("x1"), item.get("y1"), item.get("x2"), item.get("y2")
            if f_num is not None and None not in (x1, y1, x2, y2):
                gt_dict[f_num] = [x1, y1, x2, y2]
                
    video_path = SAMPLES_DIR / v_id / "drone_video.mp4"
    if not video_path.exists():
        continue
        
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idx = 0
    pred_dict = {}
    
    pbar_val = tqdm(
        total=total_frames,
        desc=f"Đánh giá Val: {v_id}",
        leave=False,
        dynamic_ncols=True
    )
    
    with torch.no_grad():
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
                
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_t = torchvision.transforms.functional.to_tensor(img_rgb).to(DEVICE).unsqueeze(0)
            outputs = eval_model(img_t)[0]
            
            keep = (outputs['scores'] > 0.30) & (outputs['labels'] == 1)
            boxes = outputs['boxes'][keep].cpu().numpy()
            
            if len(boxes) > 0:
                pred_dict[frame_idx] = boxes[0].tolist()
            frame_idx += 1
            pbar_val.update(1)
            
    cap.release()
    pbar_val.close()
    
    video_st_iou = calculate_st_iou(gt_dict, pred_dict)
    st_iou_results[v_id] = round(video_st_iou, 4)
    print(f"Video '{v_id}': STIoU = {video_st_iou:.4f}")

mean_st_iou = np.mean(list(st_iou_results.values())) if st_iou_results else 0.0

# 3. Inference Public Test & Do Hardware Profile
profiler = HardwareProfiler()
profiler.start()

TEST_DIR = ROOT_DIR / "dataset" / "public_test" / "samples"
if not TEST_DIR.exists():
    TEST_DIR = ROOT_DIR / "dataset" / "public_test"

submission_data = []
video_folders = sorted([d for d in Path(TEST_DIR).iterdir() if d.is_dir()])

print(f"\nBắt đầu Benchmark Inference trên {len(video_folders)} video kiểm thử...")

for v_folder in video_folders:
    v_id = v_folder.name
    video_path = v_folder / "drone_video.mp4"
    if not video_path.exists():
        submission_data.append({"video_id": v_id, "detections": []})
        continue

    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idx = 0
    video_bboxes = []
    
    pbar_test = tqdm(
        total=total_frames,
        desc=f"Kiểm thử: {v_id}",
        leave=False,
        dynamic_ncols=True
    )
    
    with torch.no_grad():
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            profiler.update(1)
            
            img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img_t = torchvision.transforms.functional.to_tensor(img_rgb).to(DEVICE).unsqueeze(0)
            outputs = eval_model(img_t)[0]
            
            keep = (outputs['scores'] > 0.30) & (outputs['labels'] == 1)
            boxes = outputs['boxes'][keep].cpu().numpy()
            
            for b in boxes:
                video_bboxes.append({
                    "frame": frame_idx,
                    "x1": int(b[0]), "y1": int(b[1]), "x2": int(b[2]), "y2": int(b[3])
                })
            frame_idx += 1
            pbar_test.update(1)
            
    cap.release()
    pbar_test.close()
    
    if video_bboxes:
        submission_data.append({"video_id": v_id, "detections": [{"bboxes": video_bboxes}]})
    else:
        submission_data.append({"video_id": v_id, "detections": []})

    print(f"Video '{v_id}': Phát hiện {len(video_bboxes)} BBoxes.")

# Luu Submission & Metrics
with open(RESULTS_DIR / "submission_faster_rcnn.json", "w", encoding="utf-8") as f:
    json.dump(submission_data, f, indent=2)

perf_summary = profiler.get_summary()
perf_summary["Model"] = "Faster R-CNN ResNet50-FPN V2"
perf_summary["Validation STIoU"] = round(mean_st_iou, 4)
perf_summary["Per-Video STIoU"] = st_iou_results

with open(RESULTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(perf_summary, f, indent=2)

print("\n" + "=" * 60)
print("TỔNG KẾT BENCHMARK MODEL 2 (Faster R-CNN):")
for k, v in perf_summary.items():
    print(f" - {k}: {v}")
print(f"Đường dẫn Submission File : {RESULTS_DIR / 'submission_faster_rcnn.json'}")
print(f"Đường dẫn Metrics File    : {RESULTS_DIR / 'metrics.json'}")
print("=" * 60)

Bắt đầu đánh giá STIoU trên các video Validation: {'Backpack_1', 'Backpack_0', 'Person1_0'}


Video 'Backpack_0': STIoU = 0.0813


Video 'Backpack_1': STIoU = 0.7269


Video 'Person1_0': STIoU = 0.5470

Bắt đầu Benchmark Inference trên 6 video kiểm thử...


Video 'BlackBox_0': Phát hiện 1054 BBoxes.


Video 'BlackBox_1': Phát hiện 149 BBoxes.


Video 'CardboardBox_0': Phát hiện 60 BBoxes.


Video 'CardboardBox_1': Phát hiện 901 BBoxes.


Video 'LifeJacket_0': Phát hiện 4386 BBoxes.


Video 'LifeJacket_1': Phát hiện 1896 BBoxes.

TỔNG KẾT BENCHMARK MODEL 2 (Faster R-CNN):
 - Elapsed Time (s): 2581.07
 - Throughput (FPS): 13.78
 - Peak VRAM (GB): 4.62
 - Model: Faster R-CNN ResNet50-FPN V2
 - Validation STIoU: 0.4517
 - Per-Video STIoU: {'Backpack_0': 0.0813, 'Backpack_1': 0.7269, 'Person1_0': 0.547}
Đường dẫn Submission File : /workspace/SurvivalBuddy/models/02_visdrone_faster_rcnn/results/submission_faster_rcnn.json
Đường dẫn Metrics File    : /workspace/SurvivalBuddy/models/02_visdrone_faster_rcnn/results/metrics.json


### Bảng Tổng Hợp Kết Quả Thực Nghiệm (Faster R-CNN Benchmark)

| Chỉ số Đo lường | Giá trị Đạt được | So sánh với YOLO26n Baseline | Đánh giá & Tiêu chuẩn Kỹ thuật |
| :--- | :--- | :--- | :--- |
| **Throughput (Tốc độ)** | **13.78 FPS** | Giảm 6.5x (89.98 FPS) | Không đạt chuẩn xử lý thời gian thực (>30 FPS) trên drone do cơ chế xử lý Two-Stage phức tạp. |
| **Peak VRAM Tiêu thụ** | **4.62 GB** | Tăng đáng kể (0.15 GB) | Mức chiếm dụng bộ nhớ tăng do kiến trúc ResNet50-FPN và quá trình tính toán RoIAlign, nhưng vẫn an toàn trên GPU 24GB. |
| **Validation STIoU** | **0.4517** | Tương đương (0.4569) | Điểm số tổng thể ngang bằng nhưng phân bổ độ chính xác giữa các chuỗi có sự phân hóa mạnh mẽ. |
| **Thời gian Thực thi** | **2581.07 giây (~43 phút)** | Tăng gấp 6.5 lần | Chi phí tính toán inference trên 6 video kiểm thử tăng theo độ trễ mô hình. |

---

### Phân Tích Độ Phân Hóa STIoU & Số Lượng Bounding Box

Kết quả thực nghiệm của Faster R-CNN thể hiện rõ ưu điểm và nhược điểm của kiến trúc Hai giai đoạn (Two-Stage Detector):

* **Đột phá về chất lượng Bounding Box trên các chuỗi mục tiêu rõ nét:**
  * `Backpack_1`: Đạt STIoU **0.7269** (vượt trội so với 0.5266 của YOLO26n).
  * `Person1_0`: Đạt STIoU **0.5470** (tăng mạnh so với 0.3672 của YOLO26n).
  * Cơ chế Region Proposal Network (RPN) kết hợp với RoIAlign giúp bao khít mục tiêu và giảm hiện tượng rung lắc (box jittering) giữa các frame liên tiếp.
* **Sự cố sụt giảm điểm tại `Backpack_0` (STIoU = 0.0813):**
  * Mô hình bị bắt nhầm vào một vật thể nền khác trong toàn bộ chuỗi video do đặt ngưỡng `score_thresh = 0.30` và chưa có module so khớp đặc trưng ảnh tham chiếu (Query-guided Matching).
* **Khả năng lọc nhiễu báo động giả (False Positives Reduction):**
  * `LifeJacket_0`: Giảm từ 7,210 BBoxes (YOLO26n) xuống **4,386 BBoxes**.
  * `LifeJacket_1`: Giảm từ 3,657 BBoxes (YOLO26n) xuống **1,896 BBoxes**.
  * Khâu phân loại giai đoạn hai (Fast R-CNN Head) đã loại bỏ được lượng lớn các vùng nhiễu sóng nước và phản xạ ánh sáng.

---

### Đánh Giá Khoa Học & Định Hướng Triển Khai Model 3 (RT-DETR)

* **Kết luận về Trade-off:** Faster R-CNN cải thiện độ chính xác phân định biên của vật thể nhỏ và giảm đáng kể dương tính giả trên môi trường biển, nhưng đánh đổi bằng việc sụt giảm tốc độ suy luận (13.78 FPS) và dễ bắt nhầm toàn bộ tracklet nếu thiếu ảnh mẫu đối chiếu.
* **Mục tiêu cho Model 3 (RT-DETR Drone):**
  * Ứng dụng kiến trúc Real-Time DEtection TRansformer (RT-DETR) nhằm dung hòa cả hai ưu điểm: tận dụng cơ chế Self-Attention toàn cục để bắt chính xác mục tiêu nhỏ và tối ưu hóa tốc độ xử lý đạt chuẩn real-time trên drone (>35-40 FPS).